# Task 2 — Definição do cenário de previsão

## 1. Introdução e objetivo

O projeto analisa sessões de carregamento de veículos elétricos e pretende prever se uma sessão terminará de forma anormal. A variável `is_abnormal` representa o resultado da sessão: `0` corresponde a um término normal e `1` a um término anormal, de acordo com a definição existente nos dados.

Nesta fase, a previsão é definida para o início da sessão de carregamento. Assim, o classificador deve utilizar apenas informação que esteja disponível nesse momento. Esta restrição é mesmo necessária: usar dados recolhidos durante ou após a sessão criaria um otimismo irrealista e foge totalmente ao que se pretende na prática.

## 2. Carregamento e verificação dos dados

Esta verificação permite confirmar a estrutura usada nas decisões seguintes sem repetir o EDA completo da Task 1.

In [16]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("Charging_Data_educational.csv")
REQUIRED_SOURCE_COLUMNS = {"end_cause", "order_creation_time", "start_time"}
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset não encontrado: {DATA_PATH.resolve()}")
df = pd.read_csv(DATA_PATH)
original_columns = df.columns.tolist()
df.columns = (df.columns.astype(str).str.strip().str.lower()
              .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True).str.strip("_"))
missing_required_columns = REQUIRED_SOURCE_COLUMNS.difference(df.columns)
if missing_required_columns:
    raise KeyError(f"Colunas necessárias ausentes: {sorted(missing_required_columns)}")
print(f"Ficheiro: {DATA_PATH.name}")
print(f"Dimensões antes do alvo: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print("\nTipos de dados após a normalização:")
display(df.dtypes.rename("tipo").to_frame())


Ficheiro: Charging_Data_educational.csv
Dimensões antes do alvo: 441,077 linhas x 47 colunas

Tipos de dados após a normalização:


,tipo
userid,int64
charging_post_id,int64
location_information,object
district_name,object
order_creation_time,object
transaction_power_kwh,float64
electricity_cost_yuan,float64
service_charge_yuan,float64
transaction_amount_yuan,float64
actual_payment_yuan,float64


A execução acima confirma o ficheiro, as dimensões e os tipos depois da normalização. A transformação dos nomes segue a Task 1: remoção de espaços exteriores, conversão para minúsculas, substituição de caracteres não alfanuméricos por `_` e remoção de `_` nas extremidades.

### Distribuição do alvo e valores em falta

A distribuição de `is_abnormal` confirma se o alvo é binário e mostra a quantidade de observações em cada classe. Os valores em falta são apresentados apenas para as colunas que podem influenciar a seleção de preditores; não são preenchidos nem são removidas linhas.

In [17]:
NORMAL_END_CAUSES = {"Charging ends normally", "User stops charging"}
end_cause_counts = df["end_cause"].value_counts(dropna=False)
missing_end_cause = int(df["end_cause"].isna().sum())
observed_end_causes = set(df["end_cause"].dropna().unique())
normal_end_causes = NORMAL_END_CAUSES
non_normal_end_causes = observed_end_causes - normal_end_causes
end_cause_review = pd.DataFrame({"end_cause": sorted(observed_end_causes)})
end_cause_review["observacoes"] = end_cause_review["end_cause"].map(end_cause_counts)
end_cause_review["classe_segundo_task2"] = end_cause_review["end_cause"].isin(NORMAL_END_CAUSES).map({True: 0, False: 1})
end_cause_review["estatuto"] = end_cause_review["end_cause"].isin(NORMAL_END_CAUSES).map({True: "normal", False: "não normal"})
print("Categorias observadas em end_cause e classificação segundo a regra adotada nesta Task 2:")
display(end_cause_review)
print(f"Valores em falta em end_cause: {missing_end_cause}")
print(f"Categorias normais definidas nesta Task 2: {sorted(normal_end_causes)}")
print("Categorias não normais observadas, tratadas como anormais pela regra adotada nesta Task 2:")
print(sorted(non_normal_end_causes))
print("A classificação dos motivos segue a definição metodológica estabelecida para esta Task 2.")

target = pd.Series(pd.NA, index=df.index, dtype="Int64")
known_end_cause = df["end_cause"].notna()
target.loc[known_end_cause] = (~df.loc[known_end_cause, "end_cause"].isin(NORMAL_END_CAUSES)).astype("Int64")
df["is_abnormal"] = target
print("\nDistribuição do alvo, incluindo valores em falta:")
target_distribution = (df["is_abnormal"].value_counts(dropna=False).rename_axis("is_abnormal").reset_index(name="observacoes").sort_values("is_abnormal", na_position="last").reset_index(drop=True))
target_distribution["percentagem"] = (target_distribution["observacoes"] / len(df) * 100).round(3)
display(target_distribution)
missing_summary = (df.isna().sum().rename("valores_em_falta").to_frame().assign(percentagem=lambda x: (x["valores_em_falta"] / len(df) * 100).round(3)).query("valores_em_falta > 0"))
print("Colunas com valores em falta:")
display(missing_summary)


Categorias observadas em end_cause e classificação segundo a regra adotada nesta Task 2:


,end_cause,observacoes,classe_segundo_task2,estatuto
0,AC Input Failure,99,1,não normal
1,AC switching device failure,5,1,não normal
2,BMS failure,8170,1,não normal
3,Charging ends normally,203201,0,normal
4,DC Output Fault,14813,1,não normal
5,DC switching device failure,1058,1,não normal
6,Failure of non-system control loop components ...,6941,1,não normal
7,Faulty charging connection,8539,1,não normal
8,Other faults of the charging pile,20114,1,não normal
9,Power conversion unit failure,1653,1,não normal


Valores em falta em end_cause: 0
Categorias normais definidas nesta Task 2: ['Charging ends normally', 'User stops charging']
Categorias não normais observadas, tratadas como anormais pela regra adotada nesta Task 2:
['AC Input Failure', 'AC switching device failure', 'BMS failure', 'DC Output Fault', 'DC switching device failure', 'Failure of non-system control loop components and assemblies', 'Faulty charging connection', 'Other faults of the charging pile', 'Power conversion unit failure', 'System communications failure', 'Vehicle Stake Interaction Failure', 'Vehicle charging faults', 'Vehicle communications failure']
A classificação dos motivos segue a definição metodológica estabelecida para esta Task 2.

Distribuição do alvo, incluindo valores em falta:


,is_abnormal,observacoes,percentagem
0,0,359143,81.424
1,1,81934,18.576
2,<NA>,0,0.0


Colunas com valores em falta:


,valores_em_falta,percentagem
post_hours_since_previous_completed_session,92,0.021
post_hours_since_previous_completed_abnormal,367,0.083
user_days_since_previous_completed_session,56647,12.843
user_days_since_previous_completed_abnormal,136279,30.897


A distribuição do alvo e os valores em falta são calculados diretamente do dataset. Nesta Task 2, o alvo é definido como 0 para `Charging ends normally` e `User stops charging`, e 1 para as restantes categorias; um `end_cause` ausente permaneceria ausente em vez de ser convertido silenciosamente em 1. No dataset atual não existem valores em falta em `end_cause`, pelo que a regra produz as classes observadas sem perda de linhas.


### Análise temporal de `order_creation_time` e `start_time`

A disponibilidade de `order_creation_time` não é assumida apenas porque a coluna está preenchida. Esta análise converte cópias das duas colunas para datas e horas, sem substituir os valores originais, e compara a sua ordem temporal. As percentagens da comparação usam apenas observações com ambas as datas válidas.

In [18]:
order_time = pd.to_datetime(df["order_creation_time"], errors="coerce")
start_time = pd.to_datetime(df["start_time"], errors="coerce")
valid_both = order_time.notna() & start_time.notna()
order_before = valid_both & (order_time < start_time)
order_equal = valid_both & (order_time == start_time)
order_after = valid_both & (order_time > start_time)
invalid_order_only = order_time.isna() & start_time.notna()
invalid_start_only = order_time.notna() & start_time.isna()
invalid_both = order_time.isna() & start_time.isna()
total_rows = len(df)
valid_rows = int(valid_both.sum())
order_comparison = pd.DataFrame({"relacao": ["Anterior", "Igual", "Posterior"], "observacoes": [int(order_before.sum()), int(order_equal.sum()), int(order_after.sum())]})
order_comparison["percentagem_observacoes_validas"] = (order_comparison["observacoes"] / valid_rows * 100).round(3)
print(f"Observações com ambas as datas válidas: {valid_rows:,} de {total_rows:,} ({valid_rows / total_rows * 100:.3f}% do total)")
print("Comparação temporal; percentagens sobre datas válidas:")
display(order_comparison)
invalid_dates = pd.DataFrame({"situacao": ["Apenas order_creation_time inválida/em falta", "Apenas start_time inválida/em falta", "Ambas inválidas/em falta"], "observacoes": [int(invalid_order_only.sum()), int(invalid_start_only.sum()), int(invalid_both.sum())]})
invalid_dates["percentagem_do_total"] = (invalid_dates["observacoes"] / total_rows * 100).round(3)
print("Datas inválidas ou em falta; percentagens sobre todas as linhas:")
display(invalid_dates)
print(f"Soma das situações inválidas: {int((~valid_both).sum()):,} observações")


Observações com ambas as datas válidas: 441,077 de 441,077 (100.000% do total)
Comparação temporal; percentagens sobre datas válidas:


,relacao,observacoes,percentagem_observacoes_validas
0,Anterior,178827,40.543
1,Igual,13662,3.097
2,Posterior,248588,56.359


Datas inválidas ou em falta; percentagens sobre todas as linhas:


,situacao,observacoes,percentagem_do_total
0,Apenas order_creation_time inválida/em falta,0,0.0
1,Apenas start_time inválida/em falta,0,0.0
2,Ambas inválidas/em falta,0,0.0


Soma das situações inválidas: 0 observações


A comparação mostra quantas sessões têm a criação da encomenda antes, no mesmo instante ou depois do início. Quando `order_creation_time` é posterior a `start_time`, a coluna não pode ser considerada universalmente disponível no instante inicial sem esclarecer primeiro a semântica operacional das duas datas. Por isso, a decisão permanece `Pendente de confirmação`; esta conclusão decorre da ordem observada e não de uma suposição sobre o significado da coluna.


## 3. Formulação do problema de classificação

As decisões metodológicas são as seguintes:

- **Unidade de observação:** uma sessão de carregamento de um veículo elétrico.
- **Variável-alvo:** `is_abnormal`.
- **Tipo de problema:** classificação binária.
- **Classes:** `0` — término normal; `1` — término anormal, segundo a definição existente em `end_cause`.
- **Momento da previsão:** início da sessão de carregamento.
- **Informação admissível:** informação conhecida nesse instante, incluindo contexto e histórico anterior que tenha sido calculado sem usar a sessão atual nem sessões futuras.
- **Informação não admissível:** informação observada durante ou depois da sessão, informação que depende da conclusão da sessão e qualquer variável que revele diretamente o resultado.

Em termos simples, o classificador recebe a descrição disponível no início de uma sessão e atribui essa sessão à classe normal ou anormal. A decisão não pode depender do que aconteceu depois do instante em que a previsão teria de ser produzida.

## 4. Análise e seleção dos preditores

A tabela seguinte documenta individualmente todas as colunas. As descrições são baseadas nos nomes das colunas, nos tipos observados, nos valores do dataset e na lógica documentada na Task 1. Quando a disponibilidade temporal não pode ser confirmada apenas com essa evidência, a decisão fica explicitamente pendente.

Os identificadores não são excluídos automaticamente: podem representar contexto disponível no início, mas introduzem risco de memorização e de fraca generalização para novos utilizadores ou postos. A sua utilização futura deve considerar esses riscos e a forma de validação adotada.

In [19]:
def make_decision(decision, availability, category, meaning, justification):
    valid_decisions = {"Elegível", "Excluir — variável-alvo", "Excluir — construção do alvo", "Excluir — fuga de informação", "Pendente de confirmação"}
    valid_availability = {"Sim", "Não", "Por confirmar"}
    if decision not in valid_decisions:
        raise ValueError(f"Decisão inválida: {decision}")
    if availability not in valid_availability:
        raise ValueError(f"Disponibilidade inválida: {availability}")
    return {"significado": meaning, "tipo_categoria": category, "disponibilidade_no_inicio": availability, "decisao": decision, "justificacao": justification}

metadata = {
    "userid": make_decision("Elegível", "Sim", "Identificador", "Identificador do utilizador", "O identificador está associado à sessão e pode estar disponível quando a sessão é iniciada. Mantém-se o risco de memorizar padrões individuais e de generalizar mal para novos utilizadores."),
    "charging_post_id": make_decision("Elegível", "Sim", "Identificador", "Identificador do posto de carregamento", "O identificador está associado ao posto escolhido no início. Pode memorizar padrões do posto e generalizar mal para novos postos."),
    "location_information": make_decision("Elegível", "Sim", "Localização", "Localização ou tipo de local associado à sessão", "A localização é um contexto associado ao posto e à sessão no momento da escolha, embora possa funcionar como substituto de características não observadas."),
    "district_name": make_decision("Elegível", "Sim", "Localização", "Distrito associado à sessão", "O distrito é um contexto geográfico associado ao posto no início da sessão."),
    "order_creation_time": make_decision("Pendente de confirmação", "Por confirmar", "Data/hora", "Data e hora de criação da encomenda", "No dataset, ocorre depois de Start Time em parte das observações; não é possível garantir que esteja disponível no instante da previsão."),
    "start_time": make_decision("Elegível", "Sim", "Data/hora", "Data e hora de início da sessão", "É o instante de referência da previsão e só pode ser usado como informação disponível no momento em que a previsão é produzida."),
    "end_time": make_decision("Excluir — fuga de informação", "Não", "Data/hora", "Data e hora de fim da sessão", "A data de fim só é conhecida após a evolução da sessão e revela informação posterior ao instante da previsão."),
    "payment_time": make_decision("Excluir — fuga de informação", "Não", "Data/hora", "Data e hora do pagamento", "A hora do pagamento é registada durante ou depois da sessão; a sua disponibilidade no início não é garantida e o seu valor pode depender da conclusão."),
    "transaction_power_kwh": make_decision("Excluir — fuga de informação", "Não", "Energia", "Energia transacionada/consumida", "Representa um resultado acumulado ou final da sessão, não disponível no início."),
    "electricity_cost_yuan": make_decision("Excluir — fuga de informação", "Não", "Custo", "Custo de eletricidade", "Depende da energia transacionada e de informação posterior é previsão."),
    "service_charge_yuan": make_decision("Excluir — fuga de informação", "Não", "Custo", "Taxa de serviço", "A taxa de serviço é um montante da transação e não foi documentado como disponível antes da sessão."),
    "transaction_amount_yuan": make_decision("Excluir — fuga de informação", "Não", "Custo", "Montante da transação", "O montante da transação é calculado no contexto da transação e pode depender da sua evolução ou conclusão."),
    "actual_payment_yuan": make_decision("Excluir — fuga de informação", "Não", "Custo", "Montante efetivamente pago", "O pagamento é posterior ao início e pode refletir a sessão concluída."),
    "end_cause": make_decision("Excluir — construção do alvo", "Não", "Construção do alvo", "Motivo registado para o término", "Esta é a variável usada para construir is_abnormal; não é um preditor e revela diretamente o resultado da sessão."),
    "temperature": make_decision("Elegível", "Sim", "Condição ambiental", "Temperatura estimada/previsão meteorológica", "A temperatura é considerada uma estimativa ou previsão disponível no início da sessão."),
    "relative_humidity": make_decision("Elegível", "Sim", "Condição ambiental", "Humidade relativa estimada/previsão meteorológica", "A humidade relativa é considerada uma estimativa ou previsão disponível no início da sessão."),
    "precipitation_mm": make_decision("Elegível", "Sim", "Condição ambiental", "Precipitação estimada/previsão meteorológica", "A precipitação é considerada uma estimativa ou previsão disponível no início da sessão."),
    "electricity_price": make_decision("Pendente de confirmação", "Por confirmar", "Contexto tarifário", "Preço da eletricidade", "A Task 1 não documenta se o preço é conhecido antes da sessão ou calculado a partir da transação."),
    "tariff_period": make_decision("Pendente de confirmação", "Por confirmar", "Contexto tarifário", "Período tarifário", "É necessário confirmar que o período é determinado pelo instante inicial da sessão."),
    "is_new_user": make_decision("Elegível", "Sim", "Histórico/agregado", "Indicador de utilizador novo", "O indicador está disponível no início e representa um caso cold-start quando não existe histórico anterior. Os atributos históricos foram construídos cronologicamente: utilizam apenas informação anterior ao início da sessão, excluem a sessão atual e não utilizam sessões futuras. As sessões anteriores ainda ativas não contribuem para atributos dependentes do resultado. Os indicadores de ausência de histórico permitem tratar casos cold-start; continuam a existir riscos de privacidade, memorização e generalização para novos utilizadores ou postos."),
    "user_active_sessions_at_start": make_decision("Elegível", "Sim", "Histórico/agregado", "Sessões ativas do utilizador no início", "A variável é observável no início. Os atributos históricos foram construídos cronologicamente: utilizam apenas informação anterior ao início da sessão, excluem a sessão atual e não utilizam sessões futuras. As sessões anteriores ainda ativas não contribuem para atributos dependentes do resultado. Os indicadores de ausência de histórico permitem tratar casos cold-start; continuam a existir riscos de privacidade, memorização e generalização para novos utilizadores ou postos."),
}
history_columns = [c for c in df.columns if c.startswith("post_") or c.startswith("user_previous_") or c.startswith("user_days_since_") or c.startswith("user_no_previous_")]
for column in history_columns:
    if column not in metadata:
        metadata[column] = make_decision("Elegível", "Sim", "Histórico/agregado", "Atributo histórico do posto ou utilizador", "Os atributos históricos foram construídos cronologicamente: utilizam apenas informação anterior ao início da sessão, excluem a sessão atual e não utilizam sessões futuras. As sessões anteriores ainda ativas não contribuem para atributos dependentes do resultado. Os indicadores de ausência de histórico permitem tratar casos cold-start; continuam a existir riscos de privacidade, memorização e generalização para novos utilizadores ou postos.")
rows = []
for column in df.columns:
    if column == "is_abnormal":
        info = make_decision("Excluir — variável-alvo", "Não", "Alvo", "Indicador binário de término anormal", "O indicador é o resultado que se pretende prever e não pode ser usado como preditor.")
    elif column in metadata:
        info = metadata[column]
    else:
        info = make_decision("Pendente de confirmação", "Por confirmar", "Não determinado", "O significado não pode ser confirmado apenas com a informação disponível", "É necessário confirmar o significado e a disponibilidade temporal antes de usar a coluna no início da sessão.")
    rows.append({"nome_coluna": column, **info})
predictor_decisions = pd.DataFrame(rows)
display(
    predictor_decisions.style
    .set_properties(
        subset=["justificacao"],
        **{"white-space": "normal", "max-width": "500px", "min-width": "350px"},
    )
)


,nome_coluna,significado,tipo_categoria,disponibilidade_no_inicio,decisao,justificacao
0,userid,Identificador do utilizador,Identificador,Sim,Elegível,O identificador está associado à sessão e pode estar disponível quando a sessão é iniciada. Mantém-se o risco de memorizar padrões individuais e de generalizar mal para novos utilizadores.
1,charging_post_id,Identificador do posto de carregamento,Identificador,Sim,Elegível,O identificador está associado ao posto escolhido no início. Pode memorizar padrões do posto e generalizar mal para novos postos.
2,location_information,Localização ou tipo de local associado à sessão,Localização,Sim,Elegível,"A localização é um contexto associado ao posto e à sessão no momento da escolha, embora possa funcionar como substituto de características não observadas."
3,district_name,Distrito associado à sessão,Localização,Sim,Elegível,O distrito é um contexto geográfico associado ao posto no início da sessão.
4,order_creation_time,Data e hora de criação da encomenda,Data/hora,Por confirmar,Pendente de confirmação,"No dataset, ocorre depois de Start Time em parte das observações; não é possível garantir que esteja disponível no instante da previsão."
5,transaction_power_kwh,Energia transacionada/consumida,Energia,Não,Excluir — fuga de informação,"Representa um resultado acumulado ou final da sessão, não disponível no início."
6,electricity_cost_yuan,Custo de eletricidade,Custo,Não,Excluir — fuga de informação,Depende da energia transacionada e de informação posterior é previsão.
7,service_charge_yuan,Taxa de serviço,Custo,Não,Excluir — fuga de informação,A taxa de serviço é um montante da transação e não foi documentado como disponível antes da sessão.
8,transaction_amount_yuan,Montante da transação,Custo,Não,Excluir — fuga de informação,O montante da transação é calculado no contexto da transação e pode depender da sua evolução ou conclusão.
9,actual_payment_yuan,Montante efetivamente pago,Custo,Não,Excluir — fuga de informação,O pagamento é posterior ao início e pode refletir a sessão concluída.


As variáveis históricas são consideradas elegíveis porque o enunciado informa que foram construídas cronologicamente, sem utilizar a sessão atual nem sessões futuras. As sessões anteriores ainda ativas não contribuem para atributos históricos dependentes do resultado. Os indicadores de ausência de histórico identificam casos cold-start, que devem ser interpretados com cautela. A utilização destes atributos continua a exigir atenção à privacidade, à memorização de padrões e à generalização para novos utilizadores ou postos.


### Resumo automático das decisões

O resumo seguinte é derivado exclusivamente da tabela anterior. Desta forma, os números e as listas não são introduzidos manualmente e permanecem consistentes se a estrutura do dataset for verificada novamente.

In [20]:
decision_counts = (
    predictor_decisions["decisao"]
    .value_counts()
    .rename("numero_colunas")
    .to_frame()
)
eligible_predictors = predictor_decisions.loc[
    predictor_decisions["decisao"] == "Elegível", "nome_coluna"
].tolist()
excluded_target_or_leakage = predictor_decisions.loc[
    predictor_decisions["decisao"].isin([
        "Excluir — variável-alvo",
        "Excluir — construção do alvo",
        "Excluir — fuga de informação",
    ]), "nome_coluna"
].tolist()
leakage_columns = predictor_decisions.loc[
    predictor_decisions["decisao"] == "Excluir — fuga de informação", "nome_coluna"
].tolist()
pending_columns = predictor_decisions.loc[
    predictor_decisions["decisao"] == "Pendente de confirmação", "nome_coluna"
].tolist()

print(f"Número total de colunas analisadas: {len(predictor_decisions)}")
display(decision_counts)
print("Preditores elegíveis, derivados da tabela:")
print(eligible_predictors)
print("\nVariáveis excluídas por serem alvo, construção do alvo ou fuga de informação:")
print(excluded_target_or_leakage)
print("\nVariáveis excluídas especificamente por fuga de informação:")
print(leakage_columns)
print("\nVariáveis pendentes de confirmação:")
print(pending_columns)

Número total de colunas analisadas: 48


,numero_colunas
decisao,
Elegível,36
Excluir — fuga de informação,7
Pendente de confirmação,3
Excluir — construção do alvo,1
Excluir — variável-alvo,1


Preditores elegíveis, derivados da tabela:
['userid', 'charging_post_id', 'location_information', 'district_name', 'start_time', 'temperature', 'relative_humidity', 'precipitation_mm', 'post_previous_sessions_1d', 'post_previous_abnormal_1d', 'post_previous_abnormal_rate_1d', 'post_previous_sessions_7d', 'post_previous_abnormal_7d', 'post_previous_abnormal_rate_7d', 'post_previous_sessions_30d', 'post_previous_abnormal_30d', 'post_previous_abnormal_rate_30d', 'post_hours_since_previous_completed_session', 'post_no_previous_completed_session', 'post_hours_since_previous_completed_abnormal', 'post_no_previous_completed_abnormal', 'is_new_user', 'user_previous_sessions_30d', 'user_previous_abnormal_30d', 'user_previous_abnormal_rate_30d', 'user_previous_sessions_180d', 'user_previous_abnormal_180d', 'user_previous_abnormal_rate_180d', 'user_previous_sessions_730d', 'user_previous_abnormal_730d', 'user_previous_abnormal_rate_730d', 'user_days_since_previous_completed_session', 'user_no_pre

Os atributos históricos não ficam pendentes por falta de garantia temporal: essa garantia é fornecida pelo enunciado. As variáveis tarifárias permanecem pendentes enquanto a sua disponibilidade no início não estiver confirmada.


## 5. Verificação das decisões

Esta secção funciona como um controlo de qualidade das decisões metodológicas. Antes de utilizar os preditores, é necessário confirmar que o alvo foi construído de acordo com a regra definida, que todas as colunas foram analisadas e que as listas finais não contêm variáveis proibidas ou pendentes.

O código testa cada condição de forma independente e atribui um estado: **Verificado** quando a condição foi satisfeita, **Problema** quando foi detetada uma inconsistência e **Não verificável** quando os ficheiros disponíveis não permitem confirmar a condição. Estas verificações não treinam modelos, não alteram o alvo e não substituem a validação temporal.


In [21]:
required_scenario_columns = {
    "end_cause",
    "order_creation_time",
    "start_time",
    "end_time",
    "payment_time",
}
required_columns_present = required_scenario_columns.issubset(set(df.columns))
missing_required_columns = sorted(required_scenario_columns.difference(df.columns))
columns_in_decision_table = predictor_decisions["nome_coluna"]
observed_classes = set(df["is_abnormal"].dropna().unique())

normal_rows_are_zero = (df.loc[df["end_cause"].isin(NORMAL_END_CAUSES), "is_abnormal"] == 0).all()
non_normal_rows_are_one = (df.loc[df["end_cause"].notna() & ~df["end_cause"].isin(NORMAL_END_CAUSES), "is_abnormal"] == 1).all()
missing_end_cause_stays_missing = df.loc[df["end_cause"].isna(), "is_abnormal"].isna().all()
target_has_no_missing_values = not df["is_abnormal"].isna().any()

# A regra  é verificada por componentes independentes e não apenas por uma comparação
# com a mesma série usada para construir o alvo.
target_rule_verified = normal_rows_are_zero and non_normal_rows_are_one and missing_end_cause_stays_missing
all_columns_documented_once = (
    len(columns_in_decision_table) == len(df.columns)
    and columns_in_decision_table.is_unique
    and set(columns_in_decision_table) == set(df.columns)
)
no_unexpected_decision_columns = set(columns_in_decision_table) == set(df.columns)
eligible_lists_consistent = set(eligible_predictors) == set(
    predictor_decisions.loc[predictor_decisions["decisao"] == "Elegível", "nome_coluna"]
)
leakage_lists_consistent = set(leakage_columns) == set(
    predictor_decisions.loc[predictor_decisions["decisao"] == "Excluir — fuga de informação", "nome_coluna"]
)
pending_lists_consistent = set(pending_columns) == set(
    predictor_decisions.loc[predictor_decisions["decisao"] == "Pendente de confirmação", "nome_coluna"]
)

checks = pd.DataFrame({
    "verificacao": [
        "As colunas necessárias existem no dataset normalizado",
        "Não existem valores em falta em end_cause",
        "As categorias observadas de end_cause estão apresentadas",
        "As categorias definidas como término normal são Charging ends normally e User stops charging",
        "As categorias observadas seguem a regra adotada nesta Task 2",
        "A implementação do alvo trata valores em falta sem os converter em 1",
        "A definição do alvo coincide com a regra adotada nesta Task 2",
        "O alvo contém apenas as classes 0 e 1",
        "O alvo não contém valores em falta",
        "Todas as colunas do dataset aparecem na tabela de decisões",
        "Cada coluna aparece exatamente uma vez na tabela de decisões",
        "Não existem nomes inesperados na tabela de decisões",
        "A lista de preditores elegíveis não contém duplicados",
        "Os preditores elegíveis existem no dataset",
        "is_abnormal não é um preditor",
        "end_cause não é um preditor",
        "Nenhuma variável com fuga aparece na lista elegível",
        "Variáveis pendentes não são incluídas nos preditores",
        "A lista de elegíveis corresponde às decisões registadas",
        "A lista de fuga corresponde às decisões registadas",
        "A lista de pendentes corresponde às decisões registadas",
        "A classificação dos motivos segue a definição metodológica do projeto",
    ],
    "estado": [
        "Verificado" if required_columns_present else "Problema",
        "Verificado" if missing_end_cause == 0 else "Problema",
        "Verificado" if set(end_cause_review["end_cause"]) == observed_end_causes and end_cause_review["end_cause"].is_unique else "Problema",
        "Verificado" if NORMAL_END_CAUSES.issubset(observed_end_causes) else "Problema",
        "Verificado" if target_rule_verified else "Problema",
        "Verificado" if missing_end_cause_stays_missing else "Problema",
        "Verificado" if target_rule_verified else "Problema",
        "Verificado" if observed_classes.issubset({0, 1}) else "Problema",
        "Verificado" if target_has_no_missing_values else "Problema",
        "Verificado" if set(columns_in_decision_table) == set(df.columns) else "Problema",
        "Verificado" if all_columns_documented_once else "Problema",
        "Verificado" if no_unexpected_decision_columns else "Problema",
        "Verificado" if len(eligible_predictors) == len(set(eligible_predictors)) else "Problema",
        "Verificado" if set(eligible_predictors).issubset(set(df.columns)) else "Problema",
        "Verificado" if "is_abnormal" not in eligible_predictors else "Problema",
        "Verificado" if "end_cause" not in eligible_predictors else "Problema",
        "Verificado" if not set(leakage_columns).intersection(eligible_predictors) else "Problema",
        "Verificado" if not set(pending_columns).intersection(eligible_predictors) else "Problema",
        "Verificado" if eligible_lists_consistent else "Problema",
        "Verificado" if leakage_lists_consistent else "Problema",
        "Verificado" if pending_lists_consistent else "Problema",
        "Verificado" if target_rule_verified else "Problema",
    ],
    "nota": [
        ", ".join(missing_required_columns) if missing_required_columns else "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "A regra definida para as categorias normais e anormais foi aplicada às categorias observadas.",
    ],
})
display(
    checks.style
    .set_properties(**{"white-space": "normal", "max-width": "700px", "min-width": "180px"})
    .set_table_styles([
        {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed")]},
        {"selector": "th", "props": [("white-space", "normal"), ("text-align", "left")]},
    ])
)

problems = checks.loc[checks["estado"] == "Problema", "verificacao"].tolist()
limitations = checks.loc[checks["estado"] == "Não verificável", "verificacao"].tolist()
if problems:
    print("Problemas detetados:")
    for check in problems:
        print(f"- {check}")
else:
    print("Não foram detetadas contradições nas verificações aplicáveis.")
if limitations:
    print("Limitações não verificáveis com os ficheiros disponíveis:")
    for check in limitations:
        print(f"- {check}")


,verificacao,estado,nota
0,As colunas necessárias existem no dataset normalizado,Verificado,
1,Não existem valores em falta em end_cause,Verificado,
2,As categorias observadas de end_cause estão apresentadas,Verificado,
3,As categorias definidas como término normal são Charging ends normally e User stops charging,Verificado,
4,As categorias observadas seguem a regra adotada nesta Task 2,Verificado,
5,A implementação do alvo trata valores em falta sem os converter em 1,Verificado,
6,A definição do alvo coincide com a regra adotada nesta Task 2,Verificado,
7,O alvo contém apenas as classes 0 e 1,Verificado,
8,O alvo não contém valores em falta,Verificado,
9,Todas as colunas do dataset aparecem na tabela de decisões,Verificado,


Não foram detetadas contradições nas verificações aplicáveis.


As verificações distinguem condições verificadas, problemas e limitações não verificáveis. A ausência de uma taxonomia externa de motivos não é tratada como prova de que todas as categorias são operacionalmente conhecidas; a classificação apresentada limita-se à regra adotada nesta Task 2.


## 6. Objetivos de avaliação e consequências dos erros

Considerando `is_abnormal = 1` como a classe positiva, o classificador produz uma decisão no início da sessão. Essa decisão pode desencadear ações operacionais, por isso os dois tipos principais de erro devem ser analisados antes da escolha das métricas e do limiar de decisão.

- **Falso positivo (FP):** uma sessão normal é sinalizada incorretamente como anormal. Pode gerar monitorização automática adicional, uma inspeção técnica desnecessária, uma prioridade de manutenção atribuída incorretamente ou uma intervenção junto do utilizador. As consequências podem incluir consumo de recursos, excesso de alertas, perturbação do utilizador e fadiga de alertas, levando os operadores a dar menos atenção a alertas posteriores.
- **Falso negativo (FN):** uma sessão anormal não é detetada e é classificada como normal. Neste caso, uma falha potencial pode não ser monitorizada, inspecionada ou encaminhada para manutenção. Pode perder-se a oportunidade de intervir durante a sessão ou de priorizar atempadamente um posto, um veículo ou outro componente com problemas.

No cenário proposto, um **falso negativo tende a ser o erro mais grave**, porque permite que uma sessão com término anormal passe sem sinalização. Um falso positivo também tem custos, sobretudo quando desencadeia inspeções, manutenção ou intervenções desnecessárias, mas tende a representar principalmente um consumo adicional de recursos, e não a perda de uma anomalia real. Esta prioridade assume que a finalidade principal do sistema é detetar sessões anormais e permitir uma ação de acompanhamento.

É esta lógica que deve orientar a avaliação posterior. Deve ser dada atenção à capacidade de detetar a classe anormal, por exemplo através da sensibilidade (recall) e de métricas que tornem visível o número de falsos negativos, sem ignorar a precisão e o número de falsos positivos. Um limiar de decisão mais baixo pode aumentar a deteção de sessões anormais, mas também pode aumentar os falsos positivos; um limiar mais alto tende a reduzir alertas, podendo deixar passar mais anomalias. A escolha final do limiar deve refletir o custo operacional relativo destes dois erros.

Nesta secção não são calculadas métricas, não são treinados modelos e não é escolhido um limiar. Ficam definidos os objetivos e as consequências que deverão fundamentar as decisões posteriores de avaliação e modelação.


O problema foi definido como classificação binária ao nível da sessão, com previsão no início do carregamento. A elegibilidade exige que a informação esteja disponível nesse instante e que não dependa do desenrolar ou da conclusão da sessão.

A análise temporal mostra a relação efetiva entre `order_creation_time` e `start_time`; como existem observações em que a criação ocorre depois do início, `order_creation_time` permanece pendente até a sua semântica operacional ser esclarecida. Os atributos históricos são elegíveis com base na construção cronológica descrita no enunciado, incluindo o tratamento de casos cold-start e a exclusão da sessão atual e de sessões futuras. As variáveis meteorológicas são consideradas estimativas ou previsões disponíveis no início da sessão.

As variáveis de término, pagamento, energia e custos finais, bem como `end_cause`, são excluídas por dependerem de informação posterior ou por participarem na construção do alvo. Os identificadores e a localização são elegíveis sob o critério de disponibilidade inicial, mas mantêm riscos de memorização e de generalização para novos utilizadores ou postos.

A análise dos erros estabelece que os falsos negativos tendem a ser mais graves neste cenário, porque podem deixar sessões anormais sem deteção. Os falsos positivos continuam a ser relevantes devido ao custo de monitorização, inspeção, manutenção ou intervenções desnecessárias. Esta assimetria deverá orientar a seleção posterior de métricas e do limiar de decisão, sem que essas escolhas sejam implementadas neste notebook.


## 7. Métricas, estabilidade temporal e baseline

In [22]:
# Distribuição das classes
class_counts = df["is_abnormal"].value_counts().sort_index()

# Percentagem de cada classe
class_percentages = (
    df["is_abnormal"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

print("Número de sessões por classe:")
print(class_counts)

print("\nPercentagem de sessões por classe:")
print(class_percentages.round(2))

Número de sessões por classe:
is_abnormal
0    359143
1     81934
Name: count, dtype: Int64

Percentagem de sessões por classe:
is_abnormal
0    81.42
1    18.58
Name: proportion, dtype: Float64


A variável-alvo apresenta duas classes: 359 143 sessões normais (81,42%) e 81 934 sessões anormais (18,58%). Embora o desequilíbrio entre classes não seja extremo, a classe anormal é minoritária, pelo que a accuracy, por si só, não permite avaliar adequadamente a capacidade de deteção de anomalias.

Assim, propõe-se utilizar a **PR-AUC** (`average_precision`), como métrica principal. Esta métrica permite avaliar a capacidade do modelo para identificar sessões anormais, considerando o compromisso entre a proporção de anomalias detetadas e a proporção de alertas que correspondem efetivamente a anomalias.

Como métricas complementares, serão utilizadas o recall, a precision, o F2-score, a ROC-AUC, o Matthews Correlation Coefficient (MCC) e o Brier score. Serão também contabilizados os falsos positivos e os falsos negativos por cada 1000 sessões, permitindo interpretar os resultados em termos operacionais. Nos folds temporais, a estabilidade será resumida por média, desvio-padrão e pior fold. O pior fold é o menor valor quando maior é melhor e o maior valor para o Brier score.

A escolha final da métrica principal será acompanhada pela comparação com baselines simples e pela avaliação da estabilidade dos resultados ao longo de diferentes períodos temporais. As baselines são a classe maioritária, um score aleatório proporcional à prevalência e uma regra histórica baseada nas taxas de anomalia disponíveis antes da sessão no posto e, quando necessário, no utilizador. Para um score aleatório, espera-se PR-AUC aproximadamente igual à prevalência e ROC-AUC próximo de 0,5.

### 7.1. Função para calcular as métricas
A função recebe as classes reais (y_true) e uma pontuação/probabilidade para a classe anormal (y_score). O limiar converte as pontuações em classes previstas para as métricas que dependem de uma decisão binária.
- AP (PR-AUC): métrica principal, calculada a partir das pontuações.
- Recall: fração das sessões anormais detetadas.
- Precision: fração dos alertas que são realmente anormais.
- F2: dá mais peso ao recall do que à precision.
- ROC-AUC: capacidade de ordenar sessões anormais acima das normais.
- MCC: métrica baseada nos quatro resultados da matriz de confusão.
- Brier score: erro das probabilidades; valores mais baixos são melhores.
- FP/FN por 1000: tradução dos erros para uma escala operacional.
O limiar 0.5 é apenas o valor por defeito para exemplos. Não deve ser considerado automaticamente o limiar ótimo.

In [23]:
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    fbeta_score,
    roc_auc_score,
    matthews_corrcoef,
    brier_score_loss,
    confusion_matrix,
)

def calculate_metrics(y_true, y_score, threshold=0.5):
    """Calcula métricas de classificação binária para a classe positiva 1."""
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)

    # Validações de entrada
    if y_true.ndim != 1 or y_score.ndim != 1:
        raise ValueError("y_true e y_score devem ser vetores unidimensionais.")
    if len(y_true) == 0:
        raise ValueError("y_true não pode estar vazio.")
    if len(y_true) != len(y_score):
        raise ValueError("y_true e y_score têm de ter o mesmo comprimento.")
    if not np.isin(y_true, [0, 1]).all():
        raise ValueError("y_true deve conter apenas 0 e 1.")
    if not np.isfinite(y_score).all():
        raise ValueError("y_score contém valores não finitos.")
    if ((y_score < 0) | (y_score > 1)).any():
        raise ValueError("y_score deve conter probabilidades entre 0 e 1.")
    if not np.isscalar(threshold) or not np.isfinite(threshold):
        raise ValueError("threshold deve ser um valor numérico finito.")
    if not 0 <= threshold <= 1:
        raise ValueError("threshold deve estar entre 0 e 1.")

    # Garantir que as classes são inteiras
    y_true = y_true.astype(int)

    # Converte as probabilidades em previsões binárias com base no limiar fornecido
    y_pred = (y_score >= threshold).astype(int)

    # Calcula a matriz de confusão e extrai os valores de TN, FP, FN e TP
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    # ROC-AUC não está definida se y_true tiver apenas uma classe.
    roc_auc = (
        roc_auc_score(y_true, y_score)
        if len(np.unique(y_true)) == 2 else np.nan
    )

    n = len(y_true)
    return {
        "AP (PR-AUC)": average_precision_score(y_true, y_score),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "F2-score": fbeta_score(
            y_true, y_pred, beta=2, zero_division=0
        ),
        "ROC-AUC": roc_auc,
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Brier score": brier_score_loss(y_true, y_score),
        "FP por 1000 sessões": fp / n * 1000,
        "FN por 1000 sessões": fn / n * 1000,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }

A função calculate_metrics definida nesta secção será utilizada nas secções seguintes para avaliar as diferentes abordagens de classificação. A sua definição permite calcular as mesmas métricas de forma consistente, facilitando a comparação entre as baselines e os modelos desenvolvidos.

Nesta fase, a função é apenas definida, pelo que ainda não são apresentados resultados de avaliação. As métricas serão calculadas posteriormente, utilizando as previsões de cada abordagem e os valores reais da classe is_abnormal correspondentes aos conjuntos de teste de cada fold temporal.

### 7.2. Criar folds temporais
Para avaliar as diferentes abordagens de classificação, é necessário dividir os dados em conjuntos de treino e teste. Como as sessões de carregamento ocorrem ao longo do tempo, esta divisão deve respeitar a ordem cronológica das observações.

Neste trabalho, utiliza-se o método TimeSeriesSplit, da biblioteca Scikit-learn, para criar cinco folds temporais. Em cada fold, o conjunto de treino contém observações anteriores às do conjunto de teste, sendo utilizada uma janela de treino crescente (expanding window).

Esta estratégia permite simular um cenário de utilização real, no qual as abordagens são desenvolvidas com base em dados históricos e avaliadas em sessões posteriores. Permite também analisar a estabilidade dos resultados ao longo do tempo e comparar as diferentes abordagens nos mesmos períodos.

Antes de criar os folds, é necessário preparar o DataFrame que contém as observações a utilizar na avaliação.

 - Preparação do conjunto de dados de avaliação

O DataFrame evaluation_df é criado a partir do DataFrame original df e contém as observações que serão utilizadas na avaliação das diferentes abordagens de classificação. A sua criação permite preparar os dados especificamente para a divisão temporal, sem alterar diretamente o DataFrame original.

Inicialmente, é criada uma cópia de df. Em seguida, a coluna start_time é convertida para o formato de data/hora, são removidas as observações sem data de início ou sem valor na variável-alvo is_abnormal e os dados são ordenados cronologicamente. Por fim, os índices são reorganizados para facilitar a seleção das observações em cada fold.

Esta preparação é necessária porque o método TimeSeriesSplit divide os dados de acordo com a ordem das observações. Assim, a ordenação cronológica garante que os conjuntos de treino correspondem a períodos anteriores aos respetivos conjuntos de teste.

O evaluation_df será utilizado nas secções seguintes para selecionar as observações de treino e teste de cada fold, através dos índices armazenados em temporal_folds. Desta forma, as diferentes baselines e os modelos serão avaliados nas mesmas divisões temporais, permitindo uma comparação consistente dos resultados.

In [24]:
# Preparar os dados para a avaliação temporal
evaluation_df = df.copy()

# Garantir que start_time está no formato datetime
evaluation_df["start_time"] = pd.to_datetime(
    evaluation_df["start_time"],
    errors="coerce"
)

# Remover observações sem data de início ou sem classe-alvo
evaluation_df = evaluation_df.dropna(
    subset=["start_time", "is_abnormal"]
)

# Ordenar cronologicamente
evaluation_df = (
    evaluation_df
    .sort_values("start_time")
    .reset_index(drop=True)
)

print("Número de observações:", len(evaluation_df))
print("Data inicial:", evaluation_df["start_time"].min())
print("Data final:", evaluation_df["start_time"].max())
print("Valores da variável-alvo:")
print(evaluation_df["is_abnormal"].value_counts())

Número de observações: 441077
Data inicial: 2019-12-31 21:27:03
Data final: 2021-12-31 23:52:14
Valores da variável-alvo:
is_abnormal
0    359143
1     81934
Name: count, dtype: Int64


Após a preparação dos dados, foram mantidas 441 077 sessões de carregamento, abrangendo o período entre 31 de dezembro de 2019 e 31 de dezembro de 2021.

Relativamente à distribuição da variável-alvo is_abnormal, verificou-se que 359 143 sessões (aproximadamente 81,42%) são classificadas como normais (0), enquanto 81 934 sessões (cerca de 18,58%) são classificadas como anómalas (1).

Estes resultados evidenciam um desequilíbrio entre as duas classes, uma vez que as sessões normais são significativamente mais frequentes do que as anómalas. Este aspeto deve ser considerado na avaliação das abordagens de classificação, pois um modelo que preveja maioritariamente a classe normal poderá apresentar uma elevada exatidão (accuracy) sem conseguir identificar adequadamente as sessões anómalas. Por este motivo, serão utilizadas métricas como a Average Precision (AP), o recall e o F2-score, que permitem avaliar melhor a capacidade de deteção da classe minoritária.

In [25]:
from sklearn.model_selection import TimeSeriesSplit

N_SPLITS = 5
if len(evaluation_df) <= N_SPLITS:
    raise ValueError("Não existem observações suficientes para criar os folds temporais.")

time_splitter = TimeSeriesSplit(n_splits=N_SPLITS)
temporal_folds = list(time_splitter.split(evaluation_df))

for fold_number, (train_idx, test_idx) in enumerate(temporal_folds, start=1):
    train_dates = evaluation_df.iloc[train_idx]["start_time"]
    test_dates = evaluation_df.iloc[test_idx]["start_time"]
    print(
        f"Fold {fold_number}: "
        f"treino até {train_dates.max()} | "
        f"teste de {test_dates.min()} a {test_dates.max()} | "
        f"{len(train_idx):,} treino / {len(test_idx):,} teste"
    )

Fold 1: treino até 2020-07-23 14:56:56 | teste de 2020-07-23 14:57:05 a 2020-11-02 21:43:39 | 73,517 treino / 73,512 teste
Fold 2: treino até 2020-11-02 21:43:39 | teste de 2020-11-02 21:44:16 a 2021-03-17 16:07:35 | 147,029 treino / 73,512 teste
Fold 3: treino até 2021-03-17 16:07:35 | teste de 2021-03-17 16:07:54 a 2021-07-12 14:29:41 | 220,541 treino / 73,512 teste
Fold 4: treino até 2021-07-12 14:29:41 | teste de 2021-07-12 14:30:46 a 2021-10-07 15:20:57 | 294,053 treino / 73,512 teste
Fold 5: treino até 2021-10-07 15:20:57 | teste de 2021-10-07 15:21:14 a 2021-12-31 23:52:14 | 367,565 treino / 73,512 teste


Os resultados mostram que foram criados cinco folds temporais, respeitando a ordem cronológica das sessões de carregamento. Em cada fold, o conjunto de teste contém 73 512 observações, enquanto o conjunto de treino aumenta progressivamente, passando de 73 517 observações no primeiro fold para 367 565 no quinto.

Esta estratégia de janela de treino crescente permite avaliar as diferentes abordagens em períodos temporais distintos, utilizando progressivamente mais dados históricos. Os índices dos conjuntos de treino e teste são armazenados na lista temporal_folds e serão reutilizados nas secções seguintes, garantindo que as baselines e os modelos são avaliados nas mesmas divisões temporais.

Desta forma, será possível comparar o desempenho das diferentes abordagens em condições consistentes e analisar a sua estabilidade ao longo do tempo.

### 7.3. Baseline 1 — classe maioritária
A primeira baseline consiste em prever sempre a classe mais frequente no conjunto de treino de cada fold. No conjunto de dados atual, a classe normal (0) é a maioritária, representando aproximadamente 81,42% das sessões.

O objetivo desta baseline é estabelecer uma referência simples para avaliar se as abordagens seguintes conseguem melhorar a deteção de sessões anómalas. Como todas as sessões são classificadas como normais, esta estratégia poderá apresentar uma accuracy elevada, mas não conseguirá identificar sessões anómalas, evidenciando as limitações desta métrica em problemas com classes desequilibradas.

Para a métrica Average Precision (AP), são atribuídas pontuações constantes a todas as sessões. Como estas pontuações não permitem distinguir as sessões normais das anómalas, a AP corresponde à prevalência da classe positiva no conjunto de teste.

Esta baseline permite, assim, estabelecer um ponto de comparação para as restantes abordagens e demonstrar a importância de utilizar métricas que avaliem especificamente a capacidade de deteção da classe anómala.

In [26]:
import numpy as np

majority_fold_results = []

for fold_number, (train_idx, test_idx) in enumerate(temporal_folds, start=1):
    y_train = evaluation_df.iloc[train_idx]["is_abnormal"].to_numpy()
    y_test = evaluation_df.iloc[test_idx]["is_abnormal"].to_numpy()

    majority_class = int(pd.Series(y_train).mode().iloc[0])
    # Pontuação constante: 0 se a classe maioritária for normal, 1 se for anormal.
    y_score = np.full(len(y_test), float(majority_class))

    result = calculate_metrics(y_test, y_score, threshold=0.5)
    result["fold"] = fold_number
    result["baseline"] = "Classe maioritária"
    majority_fold_results.append(result)

majority_results_df = pd.DataFrame(majority_fold_results)
display(majority_results_df)

,AP (PR-AUC),Recall,Precision,F2-score,ROC-AUC,MCC,Brier score,FP por 1000 sessões,FN por 1000 sessões,TP,TN,FP,FN,fold,baseline
0,0.187371,0.0,0.0,0.0,0.5,0.0,0.187371,0.0,187.370769,0,59738,0,13774,1,Classe maioritária
1,0.164000,0.0,0.0,0.0,0.5,0.0,0.164000,0.0,164.000435,0,61456,0,12056,2,Classe maioritária
2,0.179495,0.0,0.0,0.0,0.5,0.0,0.179495,0.0,179.494504,0,60317,0,13195,3,Classe maioritária
3,0.214441,0.0,0.0,0.0,0.5,0.0,0.214441,0.0,214.441180,0,57748,0,15764,4,Classe maioritária
4,0.171795,0.0,0.0,0.0,0.5,0.0,0.171795,0.0,171.795081,0,60883,0,12629,5,Classe maioritária


Os resultados obtidos são consistentes com o comportamento esperado da baseline da classe maioritária. Em todos os folds, a classe normal (0) é a mais frequente no conjunto de treino, pelo que todas as sessões do conjunto de teste são classificadas como normais.

O recall e o F2-score apresentam valores iguais a zero, uma vez que nenhuma sessão anómala é identificada. Consequentemente, a precision também é zero, de acordo com a convenção utilizada pelo Scikit-learn quando não existem previsões positivas. O MCC é igualmente zero, enquanto o ROC-AUC é 0,5, indicando que as pontuações constantes não permitem distinguir as sessões normais das anómalas.

A Average Precision (AP) varia entre aproximadamente 0,164 e 0,214, correspondendo à prevalência da classe anómala em cada conjunto de teste. O Brier score apresenta os mesmos valores, uma vez que a probabilidade prevista para a classe anómala é zero em todas as sessões. Relativamente às taxas de erro, a baseline apresenta zero falsos positivos por 1000 sessões, mas falha todas as sessões anómalas, originando aproximadamente 164 a 214 falsos negativos por 1000 sessões, consoante o fold.

Conclui-se que esta baseline não é adequada para a deteção de sessões anómalas, apesar de poder apresentar uma accuracy relativamente elevada devido à predominância da classe normal. A sua utilidade consiste em estabelecer uma referência simples, permitindo verificar se as abordagens seguintes conseguem identificar anomalias e melhorar o desempenho relativamente a uma estratégia que prevê sempre a classe maioritária.

### 7.4. Baseline 2 — aleatório proporcional à prevalência
Esta baseline é utilizada como uma segunda referência para avaliar o desempenho dos modelos de classificação desenvolvidos posteriormente. Ao contrário da baseline da classe maioritária, que prevê sempre a classe normal, esta abordagem atribui pontuações aleatórias às sessões, sem utilizar as suas características para identificar anomalias.

O limiar de classificação é definido com base na prevalência da classe anómala no conjunto de treino. Como as pontuações são geradas uniformemente entre 0 e 1, utiliza-se o limiar (1-p), em que (p) representa a proporção de sessões anómalas no treino. Desta forma, espera-se que a proporção de sessões sinalizadas como anómalas seja aproximadamente igual à prevalência observada no treino.

Esta baseline permite estabelecer uma referência para o desempenho de uma abordagem sem capacidade discriminativa. A Average Precision (AP) esperada é aproximadamente igual à prevalência da classe positiva, embora os resultados de cada fold possam variar devido à aleatoriedade.

A comparação com esta baseline permite avaliar se os modelos seguintes conseguem identificar sessões anómalas de forma mais eficaz do que uma abordagem aleatória. A utilização de uma semente aleatória fixa garante a reprodutibilidade dos resultados.

In [27]:
RANDOM_STATE = 42
random_fold_results = []

for fold_number, (train_idx, test_idx) in enumerate(temporal_folds, start=1):
    y_train = evaluation_df.iloc[train_idx]["is_abnormal"].to_numpy()
    y_test = evaluation_df.iloc[test_idx]["is_abnormal"].to_numpy()

    prevalence_train = float(y_train.mean())
    rng = np.random.default_rng(RANDOM_STATE + fold_number)
    y_score = rng.uniform(0, 1, size=len(y_test))
    random_threshold = 1 - prevalence_train

    result = calculate_metrics(
        y_test, y_score, threshold=random_threshold
    )
    result["fold"] = fold_number
    result["baseline"] = "Aleatório proporcional à prevalência"
    result["prevalência_treino"] = prevalence_train
    result["limiar"] = random_threshold
    random_fold_results.append(result)

random_results_df = pd.DataFrame(random_fold_results)
display(random_results_df)

,AP (PR-AUC),Recall,Precision,F2-score,ROC-AUC,MCC,Brier score,FP por 1000 sessões,FN por 1000 sessões,TP,TN,FP,FN,fold,baseline,prevalência_treino,limiar
0,0.189518,0.203717,0.192403,0.201349,0.504421,0.006415,0.332385,160.218740,149.200131,2806,47960,11778,10968,1,Aleatório proporcional à prevalência,0.197451,0.802549
1,0.161984,0.188786,0.161407,0.182591,0.495892,-0.003412,0.334703,160.858091,133.039504,2276,49631,11825,9780,2,Aleatório proporcional à prevalência,0.192411,0.807589
2,0.176534,0.178704,0.177014,0.178363,0.493942,-0.003041,0.334864,149.132114,147.418109,2358,49354,10963,10837,3,Aleatório proporcional à prevalência,0.182941,0.817059
3,0.213668,0.181045,0.214313,0.186846,0.499034,-0.000147,0.333400,142.330504,175.617586,2854,47285,10463,12910,4,Aleatório proporcional à prevalência,0.182079,0.817921
4,0.171750,0.186159,0.169246,0.182511,0.500023,-0.003262,0.334380,156.981173,139.813908,2351,49343,11540,10278,5,Aleatório proporcional à prevalência,0.188552,0.811448


Os resultados obtidos são consistentes com o comportamento esperado de uma baseline sem capacidade discriminativa. As pontuações são geradas aleatoriamente, não utilizando as características das sessões para identificar padrões associados às anomalias.

A Average Precision (AP) apresenta valores próximos da prevalência da classe anómala, embora varie entre os folds devido à aleatoriedade das pontuações e às diferenças na distribuição das classes. O ROC-AUC mantém-se próximo de 0,5 e o MCC próximo de zero, indicando que esta abordagem não consegue distinguir de forma fiável as sessões normais das anómalas.

O recall apresenta valores reduzidos, mostrando que apenas uma parte das sessões anómalas é identificada. A precision também é limitada, pelo que uma proporção considerável das sessões sinalizadas corresponde a falsos positivos. Consequentemente, esta estratégia pode originar falsos alarmes e, simultaneamente, deixar passar muitas sessões anómalas. O Brier score, próximo de 0,33, também reflete a inadequação das pontuações aleatórias como estimativas informadas das probabilidades de anomalia.

Conclui-se que esta baseline não é adequada para a deteção de sessões anómalas. No entanto, constitui uma referência útil para avaliar os modelos seguintes. Espera-se que estes apresentem uma capacidade discriminativa superior, refletida sobretudo em valores mais elevados de AP e ROC-AUC, bem como melhorias no recall, no F2-score e no MCC, mantendo um nível de falsos alarmes operacionalmente aceitável.

### 7.5. Resumir a estabilidade entre folds
Após avaliar cada baseline nos cinco folds temporais, é necessário resumir os resultados para obter uma visão global do seu desempenho e da sua consistência ao longo do tempo.

Para cada métrica, são calculadas a média e o desvio-padrão dos resultados obtidos nos diferentes folds. A média permite avaliar o desempenho médio da baseline, enquanto o desvio-padrão permite perceber quanto esse desempenho varia entre os períodos avaliados. Um desvio-padrão reduzido indica maior consistência dos resultados, embora não implique necessariamente um bom desempenho.

É também identificado o pior fold para cada métrica, permitindo analisar o período em que a baseline apresentou o resultado menos favorável. Para as métricas em que valores mais elevados são melhores (AP, recall, precision, F2-score, ROC-AUC e MCC), considera-se o valor mínimo. Para as métricas em que valores mais baixos são melhores (Brier score, falsos positivos por 1000 sessões e falsos negativos por 1000 sessões), considera-se o valor máximo.

Esta análise facilita a comparação entre as baselines e estabelece uma referência para avaliar os modelos seguintes. Assim, não se considera apenas o desempenho médio, mas também a sua variabilidade e os períodos em que a abordagem apresenta maiores limitações.

In [28]:
def summarize_fold_metrics(results_df, group_name):
    """Resume métricas por baseline, incluindo média, desvio-padrão e pior fold."""
    higher_is_better = {
        "AP (PR-AUC)", "Recall", "Precision", "F2-score", "ROC-AUC", "MCC"
    }
    metric_names = [
        "AP (PR-AUC)", "Recall", "Precision", "F2-score",
        "ROC-AUC", "MCC", "Brier score",
        "FP por 1000 sessões", "FN por 1000 sessões"
    ]

    rows = []
    for metric in metric_names:
        values = results_df[metric].dropna()
        if values.empty:
            mean_value = std_value = worst_value = np.nan
        else:
            mean_value = values.mean()
            std_value = values.std(ddof=1) if len(values) > 1 else 0.0
            worst_value = (
                values.min() if metric in higher_is_better else values.max()
            )
        rows.append({
            "baseline": group_name,
            "métrica": metric,
            "média": mean_value,
            "desvio-padrão": std_value,
            "pior_fold": worst_value,
        })
    return pd.DataFrame(rows)

# Resumo das métricas de cada baseline
summary_parts = [
    ("Classe maioritária",
     summarize_fold_metrics(majority_results_df, "Classe maioritária")),

    ("Aleatório proporcional à prevalência",
     summarize_fold_metrics(
         random_results_df,
         "Aleatório proporcional à prevalência"
     ))
]

# Apresentar uma tabela por baseline
for baseline_name, summary_df in summary_parts:
    print(f"\n{'=' * 60}")
    print(baseline_name)
    print('=' * 60)

    display(summary_df)


Classe maioritária


,baseline,métrica,média,desvio-padrão,pior_fold
0,Classe maioritária,AP (PR-AUC),0.183420,0.019401,0.164000
1,Classe maioritária,Recall,0.000000,0.000000,0.000000
2,Classe maioritária,Precision,0.000000,0.000000,0.000000
3,Classe maioritária,F2-score,0.000000,0.000000,0.000000
4,Classe maioritária,ROC-AUC,0.500000,0.000000,0.500000
5,Classe maioritária,MCC,0.000000,0.000000,0.000000
6,Classe maioritária,Brier score,0.183420,0.019401,0.214441
7,Classe maioritária,FP por 1000 sessões,0.000000,0.000000,0.000000
8,Classe maioritária,FN por 1000 sessões,183.420394,19.400966,214.441180



Aleatório proporcional à prevalência


,baseline,métrica,média,desvio-padrão,pior_fold
0,Aleatório proporcional à prevalência,AP (PR-AUC),0.182691,0.019953,0.161984
1,Aleatório proporcional à prevalência,Recall,0.187682,0.009815,0.178704
2,Aleatório proporcional à prevalência,Precision,0.182876,0.020977,0.161407
3,Aleatório proporcional à prevalência,F2-score,0.186332,0.008914,0.178363
4,Aleatório proporcional à prevalência,ROC-AUC,0.498662,0.004034,0.493942
5,Aleatório proporcional à prevalência,MCC,-0.000689,0.004193,-0.003412
6,Aleatório proporcional à prevalência,Brier score,0.333946,0.001042,0.334864
7,Aleatório proporcional à prevalência,FP por 1000 sessões,153.904124,7.975290,160.858091
8,Aleatório proporcional à prevalência,FN por 1000 sessões,149.017847,16.203074,175.617586


### 7.6. Interpretação e limitações

Os resultados das duas baselines permitem estabelecer referências para avaliar a capacidade de deteção de sessões anómalas e comparar o desempenho de futuras abordagens preditivas.

A baseline da classe maioritária apresenta uma AP (PR-AUC) média de 0,1834 e uma ROC-AUC de 0,5000, indicando ausência de capacidade discriminativa. Como prevê sempre a classe maioritária, não identifica sessões anómalas, apresentando Recall, Precision, F2-score e MCC iguais a zero. Apesar de poder apresentar uma accuracy elevada, esta estratégia não é adequada para a deteção de anomalias, uma vez que falha todas as sessões pertencentes à classe positiva. O número médio de falsos negativos é de 183,42 por cada 1000 sessões. A AP observada constitui uma referência para a avaliação de modelos, devendo ser interpretada considerando a prevalência da classe positiva em cada fold de teste.

A baseline aleatória proporcional à prevalência apresenta uma AP média de 0,1827, próxima da observada na baseline da classe maioritária, e uma ROC-AUC média de 0,4987. O MCC médio é praticamente nulo (-0,0007), o que é consistente com a ausência de uma associação relevante entre as previsões e as classes reais. O Recall médio é de 0,1877 e a Precision média é de 0,1829, valores próximos da prevalência da classe positiva. Esta abordagem regista, em média, 153,90 falsos positivos e 149,02 falsos negativos por cada 1000 sessões. Embora permita identificar algumas sessões anómalas, gera alertas sem uma capacidade discriminativa significativa. A utilização de uma semente aleatória permite reproduzir os resultados, mas a variabilidade entre execuções deve ser considerada na avaliação desta baseline.

A comparação entre as duas abordagens mostra que nenhuma apresenta uma capacidade relevante para distinguir sessões anómalas de sessões normais. A baseline da classe maioritária não deteta qualquer anomalia, enquanto a baseline aleatória identifica algumas sessões anómalas, mas sem demonstrar uma capacidade discriminativa significativa. Estes resultados reforçam a necessidade de desenvolver modelos que utilizem as características disponíveis das sessões para melhorar a qualidade das previsões.

Existem, no entanto, limitações metodológicas importantes. A avaliação deve respeitar a ordem temporal dos dados, garantindo que os modelos utilizam apenas informação disponível no momento da previsão e evitando a utilização de informação futura (data leakage). A avaliação depende também da definição de sessão anómala, da qualidade dos dados e do momento em que a previsão é efetuada.

Por fim, estas baselines constituem referências iniciais e não substituem a avaliação de modelos preditivos treinados com os preditores elegíveis e disponíveis no momento da previsão. Os modelos deverão ser avaliados segundo a mesma estratégia de validação temporal e comparados através das mesmas métricas. O limiar de classificação deverá ser escolhido sem utilizar o conjunto de teste. A escolha da abordagem final deverá considerar não só a capacidade de deteção de anomalias, mas também o equilíbrio entre falsos positivos e falsos negativos, tendo em conta as consequências operacionais de cada tipo de erro.